# 11 — Conversational RAG: Giving StoryVerse AI a Memory

*Advanced series, part 1 of 3. Picks up right where notebook 10 left off.*

Notebook 08's interactive loop worked, but it had a quiet flaw: every question was answered as if it were the first one ever asked. Watch what happens when a real conversation actually depends on what was just said.


In [ ]:
import os
import json
import faiss
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")
index = faiss.read_index(os.path.join(DATA_DIR, "storyverse_chunks.faiss"))
with open(os.path.join(DATA_DIR, "storyverse_chunks_docstore.json")) as f:
    doc_store = json.load(f)


def retrieve(query: str, top_k: int = 3, min_score: float = 0.35) -> list[dict]:
    query_vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = [{**doc_store[i], "score": float(scores[0][j])} for j, i in enumerate(indices[0])]
    return [r for r in results if r["score"] >= min_score]


def rag_answer(question: str, top_k: int = 3) -> dict:
    chunks = retrieve(question, top_k)
    if not chunks:
        return {"answer": "I don't have enough information about this in my knowledge base.", "chunks": []}
    context = "\n\n".join(f"[{c['metadata']['source']}]\n{c['content']}" for c in chunks)
    prompt = f"Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        reasoning_effort="low",
    )
    return {"answer": response.choices[0].message.content, "chunks": chunks}


## Watch it fail

A completely normal two-turn conversation: ask who Cooper's daughter is, then ask a natural follow-up about her.


In [ ]:
print("Turn 1:", rag_answer("Who is Cooper's daughter in Interstellar?")["answer"])
print()
print("Turn 2:", rag_answer("What happens to her next?")["answer"])


Turn 2 has no idea who "she" is. `rag_answer` embeds the literal string `"What happens to her next?"` — there's no Murph in that sentence, so retrieval has nothing to search for, and the model has no way to actually answer. This isn't a retrieval bug or a generation bug in the sense notebook 09 covered — it's a missing layer entirely: **conversation memory**.


## The simplest fix: remember what was said

A chat history is just a list of past turns. Keep it, and hand the whole thing to the model every time.


In [ ]:
def ask_with_history(question: str, history: list[dict]) -> str:
    messages = history + [{"role": "user", "content": question}]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


history = []
turn1 = ask_with_history("Who is Cooper's daughter in Interstellar?", history)
history += [{"role": "user", "content": "Who is Cooper's daughter in Interstellar?"}, {"role": "assistant", "content": turn1}]
print("Turn 1:", turn1)

turn2 = ask_with_history("What happens to her next?", history)
print("Turn 2:", turn2)


That fixes it for a bare model — the second call can see "Cooper's daughter" sitting right there in turn 1, so "she" resolves correctly. But look closely: this bypasses retrieval entirely. It works here because the bare model already knows the real Interstellar plot from training. Plug this same pattern into `rag_answer`, and a new problem shows up immediately.


## Where it breaks again: retrieval doesn't read history

`rag_answer` still embeds only the newest question to decide what to retrieve. Handing the model a history doesn't help retrieval find the right chunk if the *query embedding* itself never mentions Murph.


In [ ]:
def rag_answer_naive_history(question: str, history: list[dict], top_k: int = 3) -> dict:
    chunks = retrieve(question, top_k)  # still just the raw follow-up question
    if not chunks:
        return {"answer": "I don't have enough information about this in my knowledge base.", "chunks": []}
    context = "\n\n".join(f"[{c['metadata']['source']}]\n{c['content']}" for c in chunks)
    system_prompt = f"Answer using ONLY this context.\n\nContext:\n{context}"
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.1, reasoning_effort="low")
    return {"answer": response.choices[0].message.content, "chunks": chunks}


rag_history = []
r1 = rag_answer_naive_history("Who is Cooper's daughter in Interstellar?", rag_history)
rag_history += [{"role": "user", "content": "Who is Cooper's daughter in Interstellar?"}, {"role": "assistant", "content": r1["answer"]}]
print("Turn 1:", r1["answer"], "| chunks retrieved:", len(r1["chunks"]))

r2 = rag_answer_naive_history("What happens to her next?", rag_history)
print("Turn 2:", r2["answer"], "| chunks retrieved:", len(r2["chunks"]))


Turn 2 should come back empty or apologetic — that's retrieval failing silently. `"What happens to her next?"` scores too low against every chunk, the threshold filters it all out, and the model never even sees the history you so carefully passed in. Passing history to the *model* fixes generation. It does nothing for *retrieval*, because retrieval happens before the model ever sees anything.


## The real fix: rewrite the question before retrieving

This is called **history-aware retrieval** (or *query condensation*). Before embedding anything, ask the model to rewrite the follow-up into a standalone question using the conversation so far — the exact same query-rewriting idea from notebook 09, aimed at pronouns and implicit references instead of casual phrasing.


In [ ]:
def condense_question(question: str, history: list[dict]) -> str:
    if not history:
        return question

    history_text = "\n".join(f"{turn['role']}: {turn['content']}" for turn in history)
    prompt = f"""Given this conversation history and a follow-up question, rewrite the follow-up
as a standalone question that makes sense without the history. Return ONLY the rewritten question.

History:
{history_text}

Follow-up: {question}
Standalone question:"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        reasoning_effort="low",
    )
    return response.choices[0].message.content.strip()


def conversational_rag_answer(question: str, history: list[dict], top_k: int = 3) -> dict:
    standalone_question = condense_question(question, history)
    chunks = retrieve(standalone_question, top_k)

    if not chunks:
        answer = "I don't have enough information about this in my knowledge base."
    else:
        context = "\n\n".join(f"[{c['metadata']['source']}]\n{c['content']}" for c in chunks)
        prompt = f"Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion: {standalone_question}\n\nAnswer:"
        response = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.1, reasoning_effort="low",
        )
        answer = response.choices[0].message.content

    return {"answer": answer, "standalone_question": standalone_question, "chunks": chunks}


## Testing a real multi-turn conversation

Three turns, each depending on the one before it.


In [ ]:
conversation_history = []

turns = [
    "Who is Cooper's daughter in Interstellar?",
    "What happens to her next?",
    "How does he get the data to her?",
]

for question in turns:
    result = conversational_rag_answer(question, conversation_history)
    print(f"You: {question}")
    print(f"  (rewritten as: {result['standalone_question']!r})")
    print(f"StoryVerse AI: {result['answer']}")
    print()
    conversation_history += [
        {"role": "user", "content": question},
        {"role": "assistant", "content": result["answer"]},
    ]


Each `standalone_question` should read like something you could ask cold, with no history at all — "she" becomes "Murph," "he" becomes "Cooper." That rewritten question is what actually gets embedded and searched. The model still only ever sees the retrieved chunks plus one self-contained question; the conversational feel comes entirely from the rewriting step in front of it.


## The same idea, the LangChain way

LangChain packages this exact pattern as `create_history_aware_retriever`: a retriever wrapped with an LLM step that rewrites the query using chat history first.


In [ ]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage

lc_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = ChatGroq(model=MODEL, temperature=0.1, reasoning_effort="low")

chroma_store = Chroma.from_texts(
    texts=[d["content"] for d in doc_store],
    embedding=lc_embeddings,
    metadatas=[d["metadata"] for d in doc_store],
    collection_name="storyverse_conversational",
)
retriever = chroma_store.as_retriever(search_kwargs={"k": 3})

condense_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the follow-up question as a standalone question, given the chat history."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, condense_prompt)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context:\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
qa_chain = create_stuff_documents_chain(llm, answer_prompt)
conversational_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

lc_history = []
for question in turns:
    result = conversational_chain.invoke({"input": question, "chat_history": lc_history})
    print(f"You: {question}")
    print(f"StoryVerse AI: {result['answer']}")
    print()
    lc_history += [HumanMessage(content=question), AIMessage(content=result["answer"])]


Same two-step idea — rewrite, then retrieve, then answer — as `conversational_rag_answer`, just wired together with LangChain's `Runnable` pieces instead of two Python functions calling each other directly.


## Keeping more than one conversation straight

Real apps serve many users at once. Each needs its *own* history, looked up by a session ID — not one global list.


In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

session_store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


conversational_chain_with_memory = RunnableWithMessageHistory(
    conversational_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# Two different users, two different conversations, tracked independently
r1 = conversational_chain_with_memory.invoke(
    {"input": "Who does Shivudu fall in love with?"}, config={"configurable": {"session_id": "user-A"}},
)
r2 = conversational_chain_with_memory.invoke(
    {"input": "Who is Cooper's daughter?"}, config={"configurable": {"session_id": "user-B"}},
)
print("User A:", r1["answer"])
print("User B:", r2["answer"])

follow_up_a = conversational_chain_with_memory.invoke(
    {"input": "Who is she in love with, again?"}, config={"configurable": {"session_id": "user-A"}},
)
print("User A follow-up:", follow_up_a["answer"])


User A's follow-up correctly resolves against *user A's* history (Avanthika), completely unaffected by whatever User B asked about in between. That's `session_id` doing its job — one dictionary key per conversation, nothing shared or leaked between them.

(You may have noticed a deprecation warning: LangChain now points `RunnableWithMessageHistory` toward LangGraph's built-in persistence instead. The concept — a session key mapped to its own history — doesn't change, only the plumbing does. Notebook 12 uses LangGraph directly.)


## The bill always comes due eventually

Notebook 03's whole lesson — more context costs more and eventually confuses the model — didn't go away just because the context is now chat history instead of documents. A 50-turn conversation means 50 turns get resent on turn 51, every time, forever, unless something is done about it.


In [ ]:
for num_turns in [2, 10, 50, 200]:
    avg_turn_chars = 150  # a short question + a short answer
    total_chars = num_turns * avg_turn_chars
    print(f"{num_turns:>4} turns -> ~{total_chars // 4:,} tokens resent on every single new message")


Production systems handle this by summarizing older turns into a short running summary, trimming to only the last N turns, or a mix of both — the same "send only what's relevant" instinct from notebook 03, just applied to conversation history instead of a document library.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Chat history | The list of past turns in a conversation |
| History-aware retrieval | Rewriting a follow-up into a standalone question before embedding it |
| Query condensation | Another name for that same rewriting step |
| Session ID | A key that keeps one user's conversation separate from everyone else's |

Memory alone (passing history to the model) fixes generation but does nothing for retrieval, because retrieval runs first and never sees the conversation. History-aware retrieval fixes the actual gap: rewrite before you embed.

**Next up:** notebook 12 gives StoryVerse AI the ability to decide *for itself* whether a question needs retrieval at all, or something else entirely — tools, routing, and multi-step reasoning.

**Exercises:**

- Ask a 4th follow-up question that depends on turns 1 *and* 3 together — does `condense_question` handle it?
- Deliberately ask an unrelated question mid-conversation (e.g. about Baahubali) — does the standalone-question rewrite stay clean, or does it get confused by leftover Interstellar context?
- Implement a simple trim: keep only the last 4 messages in `history` before calling `condense_question` — how far back can you go before answers start breaking?
